# Exploring the names dataset

Goal: freely check the quality of the names in `data/interim/names.jsonl`.

Three exploration modes: random sampling, source audit, pattern search. The last sections display families of potential anomalies to be judged by eye — **no exclusion happens here**.

In [ ]:
import json
import random
import re
from collections import Counter
from pathlib import Path

path = Path("../data/interim/names.jsonl")  # kernel cwd = notebooks/
names = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines()]
len(names), names[:3]

## Sources overview

In [ ]:
volumes = Counter(n["source"] for n in names)
print(f"{len(names)} names, {len(volumes)} sources")
print()
for source, count in volumes.most_common(25):
    print(f"{count:>5}  {source}")

## Free exploration

Each cell draws a new sample on every run: re-running a cell is enough to get a fresh window into the dataset.

In [ ]:
N = 30  # sample size; re-run the cell for a new sample

for n in random.sample(names, min(N, len(names))):
    print(f"{n['canonical']:<45}  {n['source']}")

In [ ]:
SOURCE = "NotreDameDeParis"  # source-name fragment, case-insensitive
MAX_SHOWN = 60  # beyond that, random sampling within the source

pool = [n for n in names if SOURCE.lower() in n["source"].lower()]
print(f"{len(pool)} names in sources matching {SOURCE!r}")
print()
shown = pool if len(pool) <= MAX_SHOWN else random.sample(pool, MAX_SHOWN)
for n in sorted(shown, key=lambda n: n["canonical"].lower()):
    print(f"{n['canonical']:<45}  {n['source']}")

In [ ]:
PATTERN = r"^Saint"  # regex applied to the canonical name
LIMIT = 80

rx = re.compile(PATTERN)
hits = [n for n in names if rx.search(n["canonical"])]
print(f"{len(hits)} names matching /{PATTERN}/")
print()
for n in hits[:LIMIT]:
    print(f"{n['canonical']:<45}  {n['source']}")

## Potential anomalies

Control views, to be judged by eye. A source that shows up often here can be audited with the `SOURCE` cell above. Eventual exclusions will be decided in Phase 2 (normalize) — this notebook never modifies `names.jsonl`.

In [ ]:
# Generic / collective: probably not character proper names
GENERIC_RX = re.compile(
    r"^(?:un |une |deux |trois |plusieurs |famille |peuple |groupe )",
    re.IGNORECASE,
)
generic = [n for n in names if GENERIC_RX.search(n["canonical"])]
print(f"{len(generic)} generic / collective names")
print()
for n in generic:
    print(f"{n['canonical']:<45}  {n['source']}")

In [ ]:
# Opera roles listed in English on the French Wikipedia pages
ENGLISH_TOKENS = {
    "the",
    "of",
    "a",
    "an",
    "old",
    "man",
    "girl",
    "boy",
    "shepherd",
    "voice",
    "first",
    "second",
    "third",
    "from",
    "king",
    "queen",
    "servant",
    "soldier",
    "child",
    "milk",
    "dream",
    "bride",
    "mother",
}


def looks_english(name: str) -> bool:
    tokens = re.split(r"[\s'\u2019-]+", name.lower())
    return any(token in ENGLISH_TOKENS for token in tokens)


english = [n for n in names if looks_english(n["canonical"])]
print(f"{len(english)} likely English-language names; top sources:")
print(Counter(n["source"] for n in english).most_common(10))
print()
for n in english:
    print(f"{n['canonical']:<45}  {n['source']}")

In [ ]:
# Abnormally long names: often role lists rather than a name
LENGTH_LIMIT = 40

lengths = sorted(len(n["canonical"]) for n in names)
print(
    f"length: min {lengths[0]}, median  {lengths[len(lengths) // 2]}, max {lengths[-1]}"
)
print()
long = [n for n in names if len(n["canonical"]) > LENGTH_LIMIT]
print(f"{len(long)} names longer than {LENGTH_LIMIT} characters:")
print()
for n in sorted(long, key=lambda n: -len(n["canonical"])):
    print(f"{len(n['canonical']):>3}  {n['canonical']}  <- {n['source']}")

In [ ]:
# Social titles embedded in the name: informative for Phase 2 (normalize)
TITLES = (
    "roi",
    "reine",
    "marquis",
    "marquise",
    "duc",
    "duchesse",
    "comte",
    "comtesse",
    "vicomte",
    "vicomtesse",
    "baron",
    "baronne",
    "abbé",
    "madame",
    "mademoiselle",
    "monsieur",
    "maître",
    "saint",
    "sainte",
)
title_rx = re.compile(r"\b(?:" + "|" + ".join(TITLES) + r)\b", re.IGNORECASE)
titled = [n for n in names if title_rx.search(n["canonical"])]
print(f"{len(titled)} names containing a social title; distribution:")
print(
    Counter(
        m.lower() for n in titled for m in title_rx.findall(n["canonical"])
    ).most_common()
)
print()
for n in random.sample(titled, min(25, len(titled))):
    print(f"{n['canonical']:<45}  {n['source']}")